# 05 · Explanations: SHAP and counterfactuals

Why is each cornea ranked where it is, and what would change it? Code: `explain/shap_explain.py`, `explain/counterfactual.py`.

> **Privacy:** this notebook shows only counts, metrics, charts and study codes (R0001, Q0001), never NEBSL records. **Before committing, clear all outputs** (VS Code: `...` → *Clear All Outputs* → save).

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Make `recommendation` importable whether this runs from notebooks/, recommendation/ or ml/
ML = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "recommendation" / "config.py").exists())
sys.path.insert(0, str(ML))

import pandas as pd
from IPython.display import Image, display
from recommendation.config import load_config

pd.set_option("display.width", 160)
cfg = load_config()
DATASET = cfg["paths"]["matching_dataset"]
OUTPUTS = Path(cfg["paths"]["outputs"])
print("Dataset:", Path(DATASET).name)

## 1. Load the trained ranker

Run `uv run python -m recommendation.main ranker` first if the model file is missing.

In [ ]:
import joblib
from recommendation.matching.ranker import Ranker

saved = joblib.load(Path(cfg["paths"]["models"]) / "matching_ranker.joblib")
ranker = Ranker(saved["settings"], saved["estimator"])
pairs = pd.read_excel(DATASET, sheet_name="Ranking_Pairs")
print("model:", saved["settings"])

## 2. What changes the order of candidates (SHAP, relative to each pool)

In [ ]:
from recommendation.explain.shap_explain import explain_ranking, global_importance, shap_values

values = shap_values(ranker, pairs, background=pairs)
feat_imp, reason_imp = global_importance(values, pairs["Request_ID"])
display((reason_imp * 100).round(1).rename("% of |SHAP|").to_frame())
(feat_imp * 100).round(1).head(10).rename("% of |SHAP|").to_frame()

## 3. "Why #1?" for example requests (held-out test set)

In [ ]:
from recommendation.evaluation.split import grouped_split

sp = cfg["split"]
_, test = grouped_split(pairs, group_col=sp["group_col"], test_size=sp["test_size"], seed=sp["seed"])
examples = sorted(test["Request_ID"].unique())[:3]
explained = explain_ranking(ranker, pairs[pairs["Request_ID"].isin(examples)], background=pairs)
cols = ["Request_ID", "ML_Rank", "Candidate_Record_ID", "Cand_Tissue_Grade", "Cand_Days_In_Storage", "Label_Allocated", "Why"]
explained.sort_values(["Request_ID", "ML_Rank"]).groupby("Request_ID").head(3)[cols]

## 4. Counterfactuals: smallest realistic change that makes a cornea #1

In [ ]:
from recommendation.explain.counterfactual import counterfactuals, describe

for rid in examples:
    pool = pairs[pairs["Request_ID"] == rid]
    order = ranker.rank(pool).sort_values("ML_Rank")["Candidate_Record_ID"].head(3)
    print(rid)
    for c in order:
        print("   ", describe(counterfactuals(ranker, pool, c)))

## 5. Across all requests: could an actionable change have put the chosen cornea first?

In [ ]:
from recommendation.explain.counterfactual import summarise_misses

summ = summarise_misses(ranker, pairs)
display(pd.Series(summ["outcome"]).rename("requests").to_frame())
pd.Series(summ["actionable_kinds"]).rename("requests").to_frame()

Explanations describe what the model learned from past allocations, not clinical cause.